In [1]:
print("Hello, World!")

Hello, World!


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Data preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Baseline Models
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

# Ensemble Models
from sklearn.ensemble import (
    RandomForestRegressor, 
    GradientBoostingRegressor,
    BaggingRegressor,
    VotingRegressor,
    ExtraTreesRegressor
)

# PCA
from sklearn.decomposition import PCA

# Model evaluation
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import math

# Set seed
np.random.seed(42)

print("Libraries imported successfully")

Libraries imported successfully


In [3]:
# Load the training data
df = pd.read_csv('train_BRCpofr.csv')

# Display dataset info
print("Dataset shape:", df.shape)
print("\nFirst few rows:")
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
print("\nBasic statistics:")
print(df.describe())
df.head()

Dataset shape: (89392, 12)

First few rows:

Data types:
id                int64
gender              str
area                str
qualification       str
income              str
marital_status    int64
vintage           int64
claim_amount      int64
num_policies        str
policy              str
type_of_policy      str
cltv              int64
dtype: object

Missing values:
id                0
gender            0
area              0
qualification     0
income            0
marital_status    0
vintage           0
claim_amount      0
num_policies      0
policy            0
type_of_policy    0
cltv              0
dtype: int64

Basic statistics:
                 id  marital_status       vintage  claim_amount           cltv
count  89392.000000    89392.000000  89392.000000  89392.000000   89392.000000
mean   44696.500000        0.575488      4.595669   4351.502416   97952.828978
std    25805.391969        0.494272      2.290446   3262.359775   90613.814793
min        1.000000        0.000000 

,id,gender,area,qualification,income,marital_status,vintage,claim_amount,num_policies,policy,type_of_policy,cltv
0,1,Male,Urban,Bachelor,5L-10L,1,5,5790,More than 1,A,Platinum,64308
1,2,Male,Rural,High School,5L-10L,0,8,5080,More than 1,A,Platinum,515400
2,3,Male,Urban,Bachelor,5L-10L,1,8,2599,More than 1,A,Platinum,64212
3,4,Female,Rural,High School,5L-10L,0,7,0,More than 1,A,Platinum,97920
4,5,Male,Urban,High School,More than 10L,1,6,3508,More than 1,A,Gold,59736


In [4]:
# Data Preprocessing (Assignment 1 pipeline)
df_clean = df.copy()

# Drop ID column
df_clean.drop(columns=['id'], inplace=True)

# Target variable: cltv (Customer Lifetime Value)
target = 'cltv'
X = df_clean.drop(columns=[target])
y = df_clean[target]

print(f"Target variable: {target}")
print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nTarget statistics:")
print(y.describe())

Target variable: cltv
Features shape: (89392, 10)
Target shape: (89392,)

Target statistics:
count     89392.000000
mean      97952.828978
std       90613.814793
min       24828.000000
25%       52836.000000
50%       66396.000000
75%      103440.000000
max      724068.000000
Name: cltv, dtype: float64


In [5]:
# Identify categorical and numerical columns
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"Categorical columns: {categorical_cols}")
print(f"Numerical columns: {numerical_cols}")

# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_cols)
    ]
)

# Apply preprocessing
X_processed = preprocessor.fit_transform(X)
print(f"\nProcessed features shape: {X_processed.shape}")

Categorical columns: ['gender', 'area', 'qualification', 'income', 'num_policies', 'policy', 'type_of_policy']
Numerical columns: ['marital_status', 'vintage', 'claim_amount']

Processed features shape: (89392, 15)


In [6]:
# Train-Test Split (70-30, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.30, random_state=42
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print(f"Feature matrix shape (train): {X_train.shape}")
print(f"Feature matrix shape (test): {X_test.shape}")

Training set size: 62574
Test set size: 26818
Feature matrix shape (train): (62574, 15)
Feature matrix shape (test): (26818, 15)


In [7]:
# Baseline Model 1: Linear Regression
lr_baseline = LinearRegression()
lr_baseline.fit(X_train, y_train)
y_pred_lr_baseline = lr_baseline.predict(X_test)

rmse_lr_baseline = math.sqrt(mean_squared_error(y_test, y_pred_lr_baseline))
r2_lr_baseline = r2_score(y_test, y_pred_lr_baseline)
mae_lr_baseline = mean_absolute_error(y_test, y_pred_lr_baseline)

print("BASELINE MODEL 1: Linear Regression")
print(f"RMSE: {rmse_lr_baseline:.4f}")
print(f"R²:   {r2_lr_baseline:.4f}")
print(f"MAE:  {mae_lr_baseline:.4f}")

# Baseline Model 2: Decision Tree Regressor
dt_baseline = DecisionTreeRegressor(random_state=42, max_depth=10)
dt_baseline.fit(X_train, y_train)
y_pred_dt_baseline = dt_baseline.predict(X_test)

rmse_dt_baseline = math.sqrt(mean_squared_error(y_test, y_pred_dt_baseline))
r2_dt_baseline = r2_score(y_test, y_pred_dt_baseline)
mae_dt_baseline = mean_absolute_error(y_test, y_pred_dt_baseline)

print("BASELINE MODEL 2: Decision Tree Regressor")
print(f"RMSE: {rmse_dt_baseline:.4f}")
print(f"R²:   {r2_dt_baseline:.4f}")
print(f"MAE:  {mae_dt_baseline:.4f}")

# Store baseline metrics
baseline_metrics = {
    'Linear Regression': {
        'RMSE': rmse_lr_baseline,
        'R2': r2_lr_baseline,
        'MAE': mae_lr_baseline
    },
    'Decision Tree': {
        'RMSE': rmse_dt_baseline,
        'R2': r2_dt_baseline,
        'MAE': mae_dt_baseline
    }
}

print("\n✓ Baseline metrics stored for comparison")

BASELINE MODEL 1: Linear Regression
RMSE: 82595.0671
R²:   0.1523
MAE:  50967.5158
BASELINE MODEL 2: Decision Tree Regressor
RMSE: 84893.5558
R²:   0.1044
MAE:  51038.2623

✓ Baseline metrics stored for comparison
